<a href="https://colab.research.google.com/github/DiFedorchuk/Data_Harmonization_Project/blob/main/Harmonization_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Imports

In [6]:
pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 57.2 MB/s eta 0:00:00


In [7]:
from __future__ import annotations
import argparse
import hashlib
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional
import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process
from sentence_transformers import SentenceTransformer
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import NearestNeighbors

In [1]:
import torch

# Check if CUDA is available and set the device accordingly
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'CUDA is available! Using GPU: {torch.cuda.get_device_name(0)}')
else:
    device = torch.device('cpu')
    print('CUDA is not available. Using CPU.')


CUDA is available! Using GPU: Tesla T4


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Load the REFERENCE file with historical harmonization

In [8]:
import zipfile
import os
import shutil

zip_file_path = "/content/drive/MyDrive/Courses/Machine Learning/Harmonization_Project/REFERENCE.zip"
extract_dir = "/tmp/reference_data"
os.makedirs(extract_dir, exist_ok=True)
delta_file_path = "/content/drive/MyDrive/Courses/Machine Learning/Harmonization_Project/DELTA.xlsx"

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extract('REFERENCE.xlsx', extract_dir)
shutil.copy(delta_file_path, os.path.join(extract_dir, 'DELTA.xlsx'))
excel_file_path = os.path.join(extract_dir, 'REFERENCE.xlsx')
reference_df = pd.read_excel(excel_file_path, sheet_name=0, engine="openpyxl")

In [9]:
reference_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 292454 entries, 0 to 292453
Data columns (total 83 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   PROD_TAG                            292454 non-null  object 
 1   DATASET_NAME                        292454 non-null  object 
 2   COUNTRY                             292454 non-null  object 
 3   PROD_HIER_NUM                       292454 non-null  int64  
 4   PROD_HIER_NAME                      104464 non-null  object 
 5   PROD_HIER_LEVEL_NUM                 292453 non-null  float64
 6   PROD_HIER_LEVEL_NAME                292453 non-null  object 
 7   XO_LEVELS                           1 non-null       object 
 8   PRODUCT_LONG_DESCRIPTION            251301 non-null  object 
 9   PRODUCT_GEN_DESCRIPTION             292454 non-null  object 
 10  CATEGORY_LOCAL                      290419 non-null  object 
 11  SUB_CATEGORY_LOCAL        

#Config

In [10]:
import pathlib

"""Configuration constants for the harmonization pipeline."""
# Local columns used to build the search(CONCAT) key
LOCAL_COLUMNS = [
    "MANUFACTURER_LOCAL",
    "BRAND_LOCAL",
    "SUB_BRAND_LOCAL",
    "ITEM_DESCRIPTION_LOCAL",
]
BARCODE_COLUMN = "BAR_CODE_LOCAL"

# Target columns to predict
TARGET_COLUMNS = [
    "MANUFACTURER_HARMONIZED",
    "BRAND_HARMONIZED",
    "SUB_BRAND_HARMONIZED",
]

# Fuzzy matching thresholds
FUZZY_MATCH_MIN = 98
FUZZY_MATCH_MAX = 100

# Sentence-BERT model used for the embedding nearest-neighbor fallback
SBERT_MODEL_NAME = "all-MiniLM-L6-v2"

# KNN number to consider for the embedding
KNN_NEIGHBORS = 5

# Validation split fraction (held out by unique product group)
VALIDATION_FRACTION = 0.2
RANDOM_SEED = 42

# Path for cached embeddings
EMBEDDINGS_CACHE = pathlib.Path("./reference_embeddings.npz")

#Data

In [11]:
import re
import pandas as pd

"""Data loading and text-key construction utilities."""
def _clean_text(value) -> str:
    if pd.isna(value):
        return ""
    text = str(value).strip().upper()
    text = re.sub(r"\s+", " ", text)
    return text

def build_search_key(reference_df: pd.DataFrame, columns=None) -> pd.Series:
    """Concatenate the local columns into a single normalized text key
    used both for fuzzy matching and for Sentence-BERT embedding input"""
    columns = columns or LOCAL_COLUMNS # Changed from config.LOCAL_COLUMNS
    parts = [reference_df[col].map(_clean_text) for col in columns]
    key = parts[0]
    for part in parts[1:]:
        key = key.str.cat(part, sep=" | ")
    return key


def build_embedding_key(reference_df: pd.DataFrame, include_item_description: bool = True) -> pd.Series:
    """Build the key used by Sentence-BERT independently from fuzzy matching. To check fuzzy matching results """
    columns = LOCAL_COLUMNS # Changed from config.LOCAL_COLUMNS
    if not include_item_description:
        columns = [column for column in columns if column != "ITEM_DESCRIPTION_LOCAL"]
    return build_search_key(reference_df, columns=columns)


def normalize_barcode(series: pd.Series) -> pd.Series:
    """Normalize barcode values to a comparable string form.
    Non-numeric or missing barcodes are normalized to None so they are
    never treated as a barcode-exact-match key"""

    def _norm(v):
        if pd.isna(v):
            return None
        s = str(v).strip()
        if s == "":
            return None
        # Accept floats like 889742645.0 exported from Excel, and plain digit strings
        try:
            f = float(s)
        except ValueError:
            return None
        if not f.is_integer():
            return None
        return str(int(f))

    return series.map(_norm)


def prepare_dataframe(df: pd.DataFrame, include_item_description_for_embedding: bool = True) -> pd.DataFrame:
    """Add additional helper columns"""
    # Create a copy to avoid SettingWithCopyWarning
    df_copy = df.copy()
    df_copy["_SEARCH_KEY"] = build_search_key(df_copy)
    df_copy["_EMBEDDING_KEY"] = build_embedding_key(
        df_copy,include_item_description=include_item_description_for_embedding
    )
    df_copy["_BARCODE_KEY"] = normalize_barcode(df_copy[BARCODE_COLUMN]) # Changed from config.BARCODE_COLUMN
    return df_copy

#Embedings

In [12]:
import hashlib
from dataclasses import dataclass
from pathlib import Path
from typing import List
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors

"""Sentence-BERT embedding + cosine-similarity kNN"""
@dataclass
class NeighborResult:
    scores: np.ndarray  # cosine similarities of the k neighbors, descending
    ref_rows: pd.DataFrame  # the k neighbor rows from the reference set

class EmbeddingIndex:
    """Encodes the reference search keys with Sentence-BERT and serves
    cosine-similarity nearest-neighbor lookups for new/unseen text."""

    def __init__(
        self,
        reference: pd.DataFrame,
        model_name: str = None,
        k: int = None,
        key_column: str = "_EMBEDDING_KEY",
    ):
        self.reference = reference.reset_index(drop=True)
        self.model_name = model_name or SBERT_MODEL_NAME
        self.k = k or KNN_NEIGHBORS
        self.key_column = key_column
        self.model = SentenceTransformer(self.model_name)
        self._embeddings = None
        self._nn = None

    def fit(self, show_progress_bar: bool = True):
        texts = self.reference[self.key_column].tolist()
        embeddings = self.model.encode(
            texts,
            batch_size=64,
            show_progress_bar=show_progress_bar,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )
        self._embeddings = embeddings
        # Cosine similarity == dot product on normalized vectors, so use
        # the "cosine" metric directly for clarity/robustness.
        self._nn = NearestNeighbors(n_neighbors=min(self.k, len(embeddings)), metric="cosine")
        self._nn.fit(embeddings)
        return self

    def encode(self, texts: List[str]) -> np.ndarray:
        return self.model.encode(
            texts,
            batch_size=64,
            show_progress_bar=False,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )

    def query(self, text: str) -> NeighborResult:
        return self.query_batch([text])[0]

    def query_batch(self, texts: List[str]) -> List[NeighborResult]:
        if self._nn is None:
            raise RuntimeError("EmbeddingIndex.fit() must be called before querying.")
        query_emb = self.encode(texts)
        distances, indices = self._nn.kneighbors(query_emb)
        similarities = 1.0 - distances
        results = []
        for row_sims, row_idx in zip(similarities, indices):
            results.append(
                NeighborResult(scores=row_sims, ref_rows=self.reference.iloc[row_idx])
            )
        return results

    def _fingerprint(self) -> str:
        """Fingerprint of the reference search keys + model name, used to
        detect a stale on-disk cache (e.g. after the reference file changes)."""
        key_hash = pd.util.hash_pandas_object(self.reference[self.key_column], index=False).sum()
        raw = f"{self.model_name}:{self.key_column}:{len(self.reference)}:{key_hash}"
        return hashlib.sha256(raw.encode("utf-8")).hexdigest()

    def save(self, path=None):
        path = Path(path or EMBEDDINGS_CACHE)
        path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(path, embeddings=self._embeddings, fingerprint=self._fingerprint())

    def load(self, path=None) -> bool:
        """Load a cached embeddings file if present and still valid for the
        current reference. Returns True on success, False if the cache is
        missing, unreadable, or stale (e.g. the reference changed), so callers
        can fall back to `fit()`."""
        path = Path(path or EMBEDDINGS_CACHE)
        if not path.exists():
            return False
        try:
            data = np.load(path, allow_pickle=False)
            cached_fingerprint = str(data["fingerprint"])
            embeddings = data["embeddings"]
        except (OSError, KeyError, ValueError):
            return False
        if cached_fingerprint != self._fingerprint() or len(embeddings) != len(self.reference):
            return False
        self._embeddings = embeddings
        self._nn = NearestNeighbors(n_neighbors=min(self.k, len(embeddings)), metric="cosine")
        self._nn.fit(embeddings)
        return True

    def fit_or_load(
        self,
        path=None,
        show_progress_bar: bool = True,
        use_cache: bool = True,
        force_rebuild: bool = False,
    ):
        """Load a valid cached embedding index if available, otherwise encode
        the reference and (unless disabled) write it to the cache."""
        if use_cache and not force_rebuild and self.load(path):
            return self
        self.fit(show_progress_bar=show_progress_bar)
        if use_cache:
            self.save(path)
        return self

#Matcher

In [13]:
from dataclasses import dataclass
from typing import Optional
import pandas as pd
from rapidfuzz import process, fuzz

"""Barcode and fuzzy-text matchers against the historical reference dataset."""
@dataclass
class MatchResult:
    method: str  # "barcode", "fuzzy", or "none"
    score: float
    ref_row: Optional[pd.Series]

class BarcodeMatcher:
    """Exact-match lookup keyed on a normalized numeric barcode."""

    def __init__(self, reference: pd.DataFrame):
        valid = reference[reference["_BARCODE_KEY"].notna()]
        # Keep the *first* occurrence per barcode as the canonical harmonization.
        self._lookup = valid.drop_duplicates(subset="_BARCODE_KEY", keep="first").set_index(
            "_BARCODE_KEY"
        )

    def match(self, barcode_key: Optional[str]) -> MatchResult:
        if barcode_key is None or barcode_key not in self._lookup.index:
            return MatchResult(method="none", score=0.0, ref_row=None)
        row = self._lookup.loc[barcode_key]
        return MatchResult(method="barcode", score=100.0, ref_row=row)


class FuzzyMatcher:
    """RapidFuzz-based exact/near-exact matcher over the concatenated search key
    """
    def __init__(self, reference: pd.DataFrame, min_score: float = None, max_score: float = None):
        self.reference = reference.reset_index(drop=True)
        self.min_score = min_score if min_score is not None else FUZZY_MATCH_MIN # Changed from config.FUZZY_MATCH_MIN
        self.max_score = max_score if max_score is not None else FUZZY_MATCH_MAX # Changed from config.FUZZY_MATCH_MAX
        self._choices = self.reference["_SEARCH_KEY"].tolist()

    def match(self, search_key: str) -> MatchResult:
        if not search_key:
            return MatchResult(method="none", score=0.0, ref_row=None)
        result = process.extractOne(search_key, self._choices, scorer=fuzz.token_sort_ratio)
        if result is None:
            return MatchResult(method="none", score=0.0, ref_row=None)
        _, score, idx = result
        if self.min_score <= score <= self.max_score:
            return MatchResult(method="fuzzy", score=score, ref_row=self.reference.iloc[idx])
        return MatchResult(method="none", score=score, ref_row=None)

#Pipeline

In [14]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional
import pandas as pd

"""Full harmonization pipeline: barcode exact match -> RapidFuzz exact/near
match -> Sentence-BERT embedding nearest-neighbor fallback."""
@dataclass
class PredictionResult:
    method: str  # "barcode", "fuzzy", "embedding"
    confidence: float
    labels: Dict[str, object] = field(default_factory=dict)

class HarmonizationPipeline:
    """Combines exact/fuzzy matching with an embedding-based fallback to
    predict harmonized target columns for new rows."""

    def __init__(
        self,
        reference: pd.DataFrame,
        target_columns: List[str] = None,
        include_item_description_for_embedding: bool = True,
    ):
        self.target_columns = target_columns or TARGET_COLUMNS
        self.include_item_description_for_embedding = include_item_description_for_embedding
        self.reference = prepare_dataframe(
            reference,
            include_item_description_for_embedding=include_item_description_for_embedding,
        )
        self.barcode_matcher = BarcodeMatcher(self.reference)
        self.fuzzy_matcher = FuzzyMatcher(self.reference)
        self.embedding_index: Optional[EmbeddingIndex] = None

    def fit_embeddings(
        self,
        show_progress_bar: bool = True,
        use_cache: bool = True,
        cache_path=None,
        force_rebuild_cache: bool = False,
        model_name: str = None,
    ):
        self.embedding_index = EmbeddingIndex(self.reference, model_name=model_name)
        self.embedding_index.fit_or_load(
            path=cache_path,
            show_progress_bar=show_progress_bar,
            use_cache=use_cache,
            force_rebuild=force_rebuild_cache,
        )
        return self

    def _labels_from_row(self, row: pd.Series) -> Dict[str, object]:
        return {col: row[col] for col in self.target_columns}

    def _predict_from_neighbors(self, neighbor_result: NeighborResult) -> PredictionResult:
        # Majority-vote each target column independently among the k neighbors,
        # weighted by similarity; report the top neighbor's similarity as confidence.
        labels = {}
        for col in self.target_columns:
            votes = {}
            for score, value in zip(neighbor_result.scores, neighbor_result.ref_rows[col]):
                votes[value] = votes.get(value, 0.0) + float(score)
            labels[col] = max(votes, key=votes.get)
        confidence = float(neighbor_result.scores[0])
        return PredictionResult(method="embedding", confidence=confidence, labels=labels)

    def predict_row(self, row: pd.Series, embedding_only: bool = False) -> PredictionResult:
        # The following barcode and fuzzy matching steps are bypassed when embedding_only is True.
        if not embedding_only:
            # Barcode matching
            barcode_key = row.get("_BARCODE_KEY")
            match = self.barcode_matcher.match(barcode_key)
            if match.ref_row is not None:
                return PredictionResult(
                    method="barcode", confidence=match.score / 100.0, labels=self._labels_from_row(match.ref_row)
                )

            # Fuzzy matching
            search_key = row.get("_SEARCH_KEY", "")
            match = self.fuzzy_matcher.match(search_key)
            if match.ref_row is not None:
                return PredictionResult(
                    method="fuzzy", confidence=match.score / 100.0, labels=self._labels_from_row(match.ref_row)
                )
        else:
            search_key = row.get("_SEARCH_KEY", "")

        # Embedding search (this part runs when embedding_only is True or if barcode/fuzzy match fails)
        if self.embedding_index is None:
            raise RuntimeError(
                "No barcode/fuzzy match found and embeddings are not fitted. "
                "Call fit_embeddings() first."
            )
        embedding_key = row.get("_EMBEDDING_KEY", search_key)
        neighbor_result = self.embedding_index.query(embedding_key)
        return self._predict_from_neighbors(neighbor_result)

    def predict(self, df: pd.DataFrame, embedding_only: bool = False) -> pd.DataFrame:
        """Predict harmonized columns for every row of `df` (which must contain
        the raw LOCAL_COLUMNS/BARCODE_COLUMN, not the internal `_SEARCH_KEY`).

        If `embedding_only` is True, barcode/fuzzy matching is bypassed
        entirely so every row is resolved via the Sentence-BERT nearest-
        neighbor fallback, useful for isolating the embedding step's own
        performance. In that mode all rows are encoded in a single batch
        (rather than one-by-one), which is dramatically faster."""
        prepared = prepare_dataframe(
            df,
            include_item_description_for_embedding=self.include_item_description_for_embedding
        )

        if embedding_only:
            if self.embedding_index is None:
                raise RuntimeError(
                    "No barcode/fuzzy match found and embeddings are not fitted. "
                    "Call fit_embeddings() first."
                )
            neighbor_results = self.embedding_index.query_batch(prepared["_EMBEDDING_KEY"].tolist())
            records = []
            for neighbor_result in neighbor_results:
                result = self._predict_from_neighbors(neighbor_result)
                record = {"_METHOD": result.method, "_CONFIDENCE": result.confidence}
                record.update({f"{col}_PRED": v for col, v in result.labels.items()})
                records.append(record)
        else:
            records = []
            for _, row in prepared.iterrows():
                result = self.predict_row(row, embedding_only=False)
                record = {"_METHOD": result.method, "_CONFIDENCE": result.confidence}
                record.update({f"{col}_PRED": v for col, v in result.labels.items()})
                records.append(record)

        result_df = pd.DataFrame(records, index=df.index)
        return pd.concat([df.reset_index(drop=True), result_df.reset_index(drop=True)], axis=1)

#Predict

In [15]:
import argparse
import pandas as pd
import logging
from transformers.utils import logging as hf_logging
import os

hf_logging.set_verbosity_error()

def main():
    parser = argparse.ArgumentParser(description="Harmonize new local-value rows using the historical reference.")
    parser.add_argument("--input", required=True, help="Path to an .xlsx/.csv file with rows to harmonize")
    parser.add_argument("--output", required=True, help="Path to write the harmonized output")
    parser.add_argument("--reference", default=None, help="Path to Reference.xlsx (defaults to project root)")
    parser.add_argument("--quiet", action="store_true", help="Suppress the embedding progress bar")
    parser.add_argument(
        "--model-name",
        default=None,
        help="Sentence-transformers model name (defaults to config.SBERT_MODEL_NAME)",
    )
    parser.add_argument(
        "--cache-path",
        default=None,
        help="Optional embeddings cache path; use a separate path for each model",
    )
    parser.add_argument(
        "--embedding-without-description",
        action="store_true",
        help="Exclude ITEM_DESCRIPTION_LOCAL from the Sentence-BERT key; fuzzy matching still uses it",
    )
    parser.add_argument(
        "--no-cache",
        action="store_true",
        help="Don't read or write the on-disk embeddings cache; always re-encode the reference",
    )
    parser.add_argument(
        "--rebuild-cache",
        action="store_true",
        help="Ignore any existing embeddings cache and recompute it (still saved unless --no-cache is set)",
    )
    parser.add_argument(
        "--embedding-only",
        action="store_true",
        help="Bypass barcode/fuzzy matching entirely; resolve every row via the Sentence-BERT "
        "embedding fallback, to isolate and measure the embedding step's own behavior",
    )

    extract_dir = "/tmp/reference_data"
    colab_argv = [
        "--input", os.path.join(extract_dir, "DELTA.xlsx"),
        "--output", os.path.join(extract_dir, "harmonized_delta.xlsx"),
        "--embedding-only" # Add this flag to ensure only embedding search runs
    ]

    # If SBERT_MODEL_NAME is defined globally and is not None, pass it as an argument.
    if SBERT_MODEL_NAME is not None:
        colab_argv.extend(["--model-name", SBERT_MODEL_NAME])

    args = parser.parse_args(args=colab_argv) # Parse the constructed list of arguments

    # Make sure reference_df is available in the global scope if not loading via args.reference
    # In this notebook, reference_df is loaded in cell d6R4bE0A54XC
    pipeline = HarmonizationPipeline(
        reference_df, # Using reference_df from global scope
        include_item_description_for_embedding=not args.embedding_without_description,
    )
    pipeline.fit_embeddings(
        show_progress_bar=not args.quiet,
        use_cache=not args.no_cache,
        force_rebuild_cache=args.rebuild_cache,
        cache_path=args.cache_path,
        model_name=args.model_name,
    )

    # Load your new data from the specified input file.
    if args.input.lower().endswith(".csv"):
        new_df = pd.read_csv(args.input)
    else:
        new_df = pd.read_excel(args.input, engine="openpyxl")

    missing = [c for c in LOCAL_COLUMNS + [BARCODE_COLUMN] if c not in new_df.columns]
    if missing:
        raise ValueError(f"Input file is missing required columns: {missing}")

    result = pipeline.predict(new_df, embedding_only=args.embedding_only)

    # Save results to the specified output file.
    if args.output.lower().endswith(".csv"):
        result.to_csv(args.output, index=False)
    else:
        result.to_excel(args.output, index=False)

    print(f"Harmonized {len(result)} rows -> {args.output}")
    print(result["_METHOD"].value_counts())


if __name__ == "__main__":
    main()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4570 [00:00<?, ?it/s]

Harmonized 394 rows -> /tmp/reference_data/harmonized_delta.xlsx
_METHOD
embedding    394
Name: count, dtype: int64


#Evaluate

In [16]:
import argparse
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import precision_recall_fscore_support

"""Held-out validation of the harmonization pipeline.

Splits the reference data by unique product "groups" (distinct local
manufacturer/brand/sub-brand/description/barcode combinations) so that
identical products don't leak between train and validation, fits the
pipeline on the train split, and reports per-target-column and per-method
accuracy on the validation split.
"""

def split_reference(df: pd.DataFrame, validation_fraction: float = None, seed: int = None):
    validation_fraction = validation_fraction or VALIDATION_FRACTION
    seed = seed if seed is not None else RANDOM_SEED
    groups = build_search_key(df)
    splitter = GroupShuffleSplit(n_splits=1, test_size=validation_fraction, random_state=seed)
    train_idx, val_idx = next(splitter.split(df, groups=groups))
    return df.iloc[train_idx].reset_index(drop=True), df.iloc[val_idx].reset_index(drop=True)


def evaluate(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    show_progress_bar: bool = True,
    embedding_only: bool = False,
    include_item_description_for_embedding: bool = True,
    model_name: str = None,
) -> dict:
    pipeline = HarmonizationPipeline(
        train_df, include_item_description_for_embedding=include_item_description_for_embedding
    )
    # Each evaluation run trains on a different reference subset (varies by
    # split/seed), so the shared embeddings cache doesn't apply here.
    pipeline.fit_embeddings(show_progress_bar=show_progress_bar, use_cache=False, model_name=model_name)

    predictions = pipeline.predict(val_df, embedding_only=embedding_only)

    report = {"n_validation_rows": len(val_df)}
    report["method_counts"] = predictions["_METHOD"].value_counts().to_dict()

    overall_correct = np.ones(len(predictions), dtype=bool)
    for col in pipeline.target_columns:
        correct = predictions[col].astype(str) == predictions[f"{col}_PRED"].astype(str)
        report[f"{col}_accuracy"] = float(correct.mean())
        overall_correct &= correct.to_numpy()

        # Break down accuracy by matching method for this column.
        by_method = {}
        for method, group in predictions.groupby("_METHOD"):
            g_correct = group[col].astype(str) == group[f"{col}_PRED"].astype(str)
            by_method[method] = {"n": int(len(group)), "accuracy": float(g_correct.mean())}
        report[f"{col}_accuracy_by_method"] = by_method

        # Macro-averaged precision/recall/F1 treat each distinct label
        # equally regardless of how common it is, exposing performance on
        # rare brands/sub-brands that plain accuracy can hide.
        y_true = predictions[col].astype(str)
        y_pred = predictions[f"{col}_PRED"].astype(str)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="macro", zero_division=0
        )
        report[f"{col}_precision_macro"] = float(precision)
        report[f"{col}_recall_macro"] = float(recall)
        report[f"{col}_f1_macro"] = float(f1)

        # Confidence calibration: does higher reported confidence actually
        # correspond to a higher chance of being correct? Also report the
        # Brier score (mean squared error between confidence and
        # correctness), where lower is better-calibrated.
        report[f"{col}_confidence_calibration"] = _confidence_calibration(
            predictions["_CONFIDENCE"], correct
        )
        report[f"{col}_brier_score"] = float(np.mean((predictions["_CONFIDENCE"].to_numpy() - correct.to_numpy()) ** 2))

    report["all_targets_exact_match_accuracy"] = float(overall_correct.mean())
    report["confidence_mean"] = float(predictions["_CONFIDENCE"].mean())
    report["confidence_median"] = float(predictions["_CONFIDENCE"].median())
    return report, predictions


def _confidence_calibration(confidence: pd.Series, correct: pd.Series, n_bins: int = 5) -> list:
    """Bucket predictions into confidence bins and report the actual
    accuracy observed in each bin, so you can see whether e.g. "0.9+
    confidence" predictions really are right ~90%+ of the time."""
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_idx = np.clip(np.digitize(confidence, bins, right=True) - 1, 0, n_bins - 1)
    rows = []
    for b in range(n_bins):
        mask = bin_idx == b
        n = int(mask.sum())
        rows.append(
            {
                "confidence_range": f"[{bins[b]:.2f}, {bins[b + 1]:.2f}]",
                "n": n,
                "accuracy": float(correct[mask].mean()) if n > 0 else None,
            }
        )
    return rows


def main():
    parser = argparse.ArgumentParser(description="Evaluate the harmonization pipeline.")
    parser.add_argument("--reference", default=None, help="Path to Reference.xlsx")
    parser.add_argument("--validation-fraction", type=float, default=VALIDATION_FRACTION)
    parser.add_argument("--seed", type=int, default=RANDOM_SEED)
    parser.add_argument("--output-csv", default=None, help="Optional path to dump predictions as CSV")
    parser.add_argument("--quiet", action="store_true", help="Suppress the embedding progress bar")
    parser.add_argument(
        "--embedding-only",
        action="store_true",
        help="Bypass barcode/fuzzy matching entirely; resolve every row via the Sentence-BERT "
        "embedding fallback, to isolate and measure the embedding step's own accuracy",
    )
    parser.add_argument(
        "--embedding-without-description",
        action="store_true",
        help="Exclude ITEM_DESCRIPTION_LOCAL from the Sentence-BERT key; fuzzy matching still uses it",
    )
    parser.add_argument(
        "--model-name",
        default=None,
        help="Sentence-transformers model name (defaults to config.SBERT_MODEL_NAME)",
    )
    # In a Colab environment, manually set arguments to bypass command-line parsing.
    args = parser.parse_args(args=[]) # Parse an empty list of arguments

    # Set default values for Colab execution
    args.reference = None
    args.validation_fraction = VALIDATION_FRACTION # Use global VALIDATION_FRACTION
    args.seed = RANDOM_SEED # Use global RANDOM_SEED
    args.output_csv = "/tmp/reference_data/validation_predictions.xlsx" # Save predictions to an Excel file
    args.quiet = False
    args.embedding_only = True # Changed to True to run only embedding search
    args.embedding_without_description = False
    args.model_name = SBERT_MODEL_NAME # Use global SBERT_MODEL_NAME

    df = reference_df # Use the globally available reference_df

    train_df, val_df = split_reference(df, args.validation_fraction, args.seed)
    print(f"Train rows: {len(train_df)}  Validation rows: {len(val_df)}")

    report, predictions = evaluate(
        train_df,
        val_df,
        show_progress_bar=not args.quiet,
        embedding_only=args.embedding_only,
        include_item_description_for_embedding=not args.embedding_without_description,
        model_name=args.model_name,
    )

    print("\n=== Validation report ===")
    print(f"Validation rows: {report['n_validation_rows']}")
    print(f"Match method counts: {report['method_counts']}")
    for col in TARGET_COLUMNS:
        print(f"\n{col} accuracy: {report[f'{col}_accuracy']:.4f}")
        for method, stats in report[f"{col}_accuracy_by_method"].items():
            print(f"  - {method:>10}: n={{stats['n']:>5}}  accuracy={{stats['accuracy']:.4f}}")
        print(
            f"  macro precision/recall/F1: "
            f"{report[f'{col}_precision_macro']:.4f} / "
            f"{report[f'{col}_recall_macro']:.4f} / "
            f"{report[f'{col}_f1_macro']:.4f}"
        )
        print(f"  Brier score (confidence calibration, lower=better): {report[f'{col}_brier_score']:.4f}")
        print("  Confidence calibration (reported confidence vs. actual accuracy):")
        for bucket in report[f"{col}_confidence_calibration"]:
            acc = f"{bucket['accuracy']:.4f}" if bucket["accuracy"] is not None else "n/a"
            print(f"    - {bucket['confidence_range']:>14}: n={{bucket['n']:>5}}  accuracy={acc}")
    print(f"\nAll-targets exact-match accuracy: {report['all_targets_exact_match_accuracy']:.4f}")
    print(f"Confidence mean/median: {report['confidence_mean']:.4f} / {report['confidence_median']:.4f}")

    if args.output_csv:
        if args.output_csv.lower().endswith(".csv"):
            predictions.to_csv(args.output_csv, index=False)
        else:
            predictions.to_excel(args.output_csv, index=False)
        print(f"\nPredictions written to {args.output_csv}")


if __name__ == "__main__":
    main()

Train rows: 235960  Validation rows: 56494


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/3687 [00:00<?, ?it/s]


=== Validation report ===
Validation rows: 56494
Match method counts: {'embedding': 56494}

MANUFACTURER_HARMONIZED accuracy: 0.9793
  -  embedding: n={stats['n']:>5}  accuracy={stats['accuracy']:.4f}
  macro precision/recall/F1: 0.9204 / 0.9165 / 0.9131
  Brier score (confidence calibration, lower=better): 0.0203
  Confidence calibration (reported confidence vs. actual accuracy):
    -   [0.00, 0.20]: n={bucket['n']:>5}  accuracy=n/a
    -   [0.20, 0.40]: n={bucket['n']:>5}  accuracy=0.0000
    -   [0.40, 0.60]: n={bucket['n']:>5}  accuracy=0.5000
    -   [0.60, 0.80]: n={bucket['n']:>5}  accuracy=0.7066
    -   [0.80, 1.00]: n={bucket['n']:>5}  accuracy=0.9871

BRAND_HARMONIZED accuracy: 0.9702
  -  embedding: n={stats['n']:>5}  accuracy={stats['accuracy']:.4f}
  macro precision/recall/F1: 0.8124 / 0.8137 / 0.8054
  Brier score (confidence calibration, lower=better): 0.0264
  Confidence calibration (reported confidence vs. actual accuracy):
    -   [0.00, 0.20]: n={bucket['n']:>5}  